# Day 8 — Explainable Vision
## RetailMind AI Increment

### Objective

Make the existing Day-7 product classifier explainable using Grad-CAM.

Workflow:

`Product Image → Day-7 Classifier → Prediction → Grad-CAM → Heatmap → Explanation Overlay`

There is **no new model training** in this increment.

## 1. Import Libraries

TensorFlow / Keras performs model loading, prediction, gradient calculation and image processing. Matplotlib is used only for displaying and overlaying the heatmap.

In [ ]:
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

print('TensorFlow version:', tf.__version__)

## 2. Configuration

Update `MODEL_PATH` so it points to the model saved by the Day-7 transfer-learning notebook.

In [ ]:
PROJECT_ROOT = Path.cwd()

MODEL_PATH = PROJECT_ROOT / 'day7_best.keras'

DATASET_ROOT = PROJECT_ROOT / 'data' / 'fashion-product-images-small'
IMAGES_DIR = DATASET_ROOT / 'images'

OUTPUT_DIR = PROJECT_ROOT / 'artifacts' / 'day8'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Model path:', MODEL_PATH)
print('Image directory:', IMAGES_DIR)

## 3. Load the Day-7 Classifier

No new model is trained. We reuse the existing transfer-learning classifier.

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Day-7 model not found: {MODEL_PATH}. '
        'Update MODEL_PATH to your saved Day-7 .keras model.'
    )

model = keras.models.load_model(MODEL_PATH)

print('Day-7 model loaded successfully.')
model.summary()

## 4. Class Names

Use the **same class order used when the Day-7 classifier was trained**.

Replace the example list below with your exact Day-7 class names.

In [ ]:
CLASS_NAMES = [
    'Accessories',
    'Apparel',
    'Footwear',
    'Free Items',
    'Personal Care',
    'Sporting Goods'
]

print('Classes:', CLASS_NAMES)
print('Number of classes:', len(CLASS_NAMES))

## 5. Check Model Input and Output

The number of prediction outputs must match the number of class names.

In [ ]:
print('Model input shape :', model.input_shape)
print('Model output shape:', model.output_shape)

if model.output_shape[-1] != len(CLASS_NAMES):
    print('WARNING: CLASS_NAMES does not match the model output size.')
    print('Replace CLASS_NAMES with the exact Day-7 class ordering.')

## 6. Find the Final Convolutional Layer

Grad-CAM requires convolutional feature maps.

This helper searches backwards through the model for the last 4-dimensional feature output.

In [ ]:
def find_last_conv_layer(model):
    for layer in reversed(model.layers):
        try:
            output_shape = layer.output.shape
        except Exception:
            continue

        if len(output_shape) == 4:
            return layer.name

    raise ValueError('No 4D convolutional layer was found.')


LAST_CONV_LAYER = find_last_conv_layer(model)

print('Last convolutional layer:', LAST_CONV_LAYER)

### Nested transfer-learning models

If your MobileNetV2 backbone appears as a nested Keras model and the automatic search does not reach its convolutional features, set `LAST_CONV_LAYER` manually to the final convolutional layer used in your Day-7 architecture.

## 7. Image Loading Function

The image is resized to the input size expected by the Day-7 model.

In [ ]:
IMAGE_HEIGHT = model.input_shape[1]
IMAGE_WIDTH = model.input_shape[2]


def load_product_image(image_path):
    image = tf.io.read_file(str(image_path))
    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(
        image,
        [IMAGE_HEIGHT, IMAGE_WIDTH]
    )

    image = tf.cast(image, tf.float32)

    display_image = tf.clip_by_value(image / 255.0, 0, 1)

    return image, display_image

## 8. Model Preprocessing

This must match Day 7.

If your Day-7 model already contains resizing/rescaling/MobileNetV2 preprocessing **inside the saved model**, return the image unchanged.

If preprocessing was outside the model, modify this function to match Day 7 exactly.

In [ ]:
def prepare_for_model(image):
    # Default assumption: Day-7 preprocessing was saved inside the model.
    return image


## 9. Grad-CAM Model

The Grad-CAM helper returns both:

- convolutional feature maps; and
- final class predictions.

In [ ]:
last_conv_layer = model.get_layer(LAST_CONV_LAYER)

grad_model = keras.Model(
    inputs=model.inputs,
    outputs=[last_conv_layer.output, model.output]
)

print('Grad-CAM model created.')

## 10. Grad-CAM Heatmap Function

Grad-CAM asks:

> Which spatial regions in the final convolutional feature map had the strongest influence on the selected class prediction?

In [ ]:
def make_gradcam_heatmap(image_batch, pred_index=None):

    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(
            image_batch,
            training=False
        )

        if pred_index is None:
            pred_index = int(tf.argmax(predictions[0]))

        class_score = predictions[:, pred_index]

    gradients = tape.gradient(
        class_score,
        conv_output
    )

    pooled_gradients = tf.reduce_mean(
        gradients,
        axis=(0, 1, 2)
    )

    conv_output = conv_output[0]

    heatmap = tf.reduce_sum(
        conv_output * pooled_gradients,
        axis=-1
    )

    heatmap = tf.maximum(heatmap, 0)

    heatmap = heatmap / (
        tf.reduce_max(heatmap) + 1e-8
    )

    return heatmap.numpy(), pred_index, predictions[0].numpy()

## 11. Prediction Function

In [ ]:
def predict_product(image_path):
    image, display_image = load_product_image(image_path)

    model_image = prepare_for_model(image)
    image_batch = tf.expand_dims(model_image, axis=0)

    predictions = model.predict(
        image_batch,
        verbose=0
    )[0]

    predicted_index = int(np.argmax(predictions))
    predicted_class = CLASS_NAMES[predicted_index]
    confidence = float(predictions[predicted_index])

    return {
        'class_index': predicted_index,
        'class_name': predicted_class,
        'confidence': confidence,
        'probabilities': predictions,
        'display_image': display_image,
        'model_image': model_image
    }

## 12. Grad-CAM Overlay Function

The heatmap is resized to the product-image dimensions and blended using Matplotlib.

In [ ]:
def show_gradcam_overlay(
    display_image,
    heatmap,
    class_name,
    confidence,
    alpha=0.45
):

    heatmap_resized = tf.image.resize(
        heatmap[..., None],
        [display_image.shape[0], display_image.shape[1]]
    ).numpy().squeeze()

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))

    axes[0].imshow(display_image)
    axes[0].set_title('Original image')
    axes[0].axis('off')

    axes[1].imshow(display_image)
    axes[1].imshow(
        heatmap_resized,
        cmap='jet',
        alpha=alpha
    )

    axes[1].set_title(
        f'{class_name}\nconfidence: {confidence:.3f}'
    )
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    return heatmap_resized

# RetailMind Explainable Vision Function

The classifier and Grad-CAM explanation are now combined behind one function.

## 13. Combined `explain_prediction()` Function

In [ ]:
def explain_prediction(image_path, alpha=0.45):

    result = predict_product(image_path)

    image_batch = tf.expand_dims(
        result['model_image'],
        axis=0
    )

    heatmap, predicted_index, probabilities = make_gradcam_heatmap(
        image_batch,
        pred_index=result['class_index']
    )

    heatmap_resized = show_gradcam_overlay(
        result['display_image'],
        heatmap,
        result['class_name'],
        result['confidence'],
        alpha=alpha
    )

    explanation = {
        'image_path': str(image_path),
        'predicted_class': result['class_name'],
        'class_index': predicted_index,
        'confidence': round(result['confidence'], 4),
        'heatmap': heatmap_resized
    }

    print('Prediction :', explanation['predicted_class'])
    print('Confidence :', explanation['confidence'])

    return explanation

## 14. Select a Product Image

Replace the example ID with an image that exists in your Fashion Product Images dataset.

In [ ]:
IMAGE_PATH = IMAGES_DIR / '1163.jpg'

if not IMAGE_PATH.exists():
    available_images = sorted(IMAGES_DIR.glob('*.jpg'))

    if not available_images:
        raise FileNotFoundError(
            f'No JPG product images were found in {IMAGES_DIR}'
        )

    IMAGE_PATH = available_images[0]

print('Using:', IMAGE_PATH)

## 15. Explain One Prediction

In [ ]:
explanation = explain_prediction(IMAGE_PATH)

## 16. Explain Several Product Images

Looking at several examples is more useful than trusting a single Grad-CAM visualization.

In [ ]:
sample_paths = sorted(IMAGES_DIR.glob('*.jpg'))[:3]

explanations = []

for path in sample_paths:
    print('\nImage:', path.name)
    result = explain_prediction(path)
    explanations.append(result)

# How to Interpret Grad-CAM

The red/yellow regions represent areas that contributed more strongly to the selected prediction.

### Good sign

The highlighted region overlaps the actual product, such as:

- shoe body;
- shirt shape;
- sleeve area;
- bag body;
- watch face or strap.

### Warning sign

If the strongest activation is repeatedly on:

- background color;
- image borders;
- watermark;
- photography setup; or
- irrelevant regions,

the classifier may be using a spurious visual shortcut.

Grad-CAM is an interpretation aid, not proof that a feature caused the prediction.

## 17. Save the Grad-CAM Heatmap

In [ ]:
heatmap = explanation['heatmap']

heatmap_uint8 = tf.cast(
    tf.clip_by_value(heatmap * 255.0, 0, 255),
    tf.uint8
)

heatmap_path = OUTPUT_DIR / 'gradcam_heatmap.png'

tf.io.write_file(
    str(heatmap_path),
    tf.io.encode_png(heatmap_uint8[..., None])
)

print('Saved heatmap:', heatmap_path)

## 18. Final Validation

In [ ]:
assert model is not None
assert LAST_CONV_LAYER is not None
assert explanation['predicted_class'] in CLASS_NAMES
assert 0.0 <= explanation['confidence'] <= 1.0
assert explanation['heatmap'].ndim == 2
assert heatmap_path.exists()

print('All Day 8 explainable-vision checks passed.')

# Day 8 Summary

In this RetailMind increment we:

- reused the existing Day-7 transfer-learning classifier;
- loaded and prepared Fashion Product Images;
- identified the final convolutional feature map;
- calculated gradients for the predicted class;
- created Grad-CAM heatmaps;
- resized heatmaps to the product-image size;
- overlaid attention regions using Matplotlib;
- inspected several predictions for possible visual shortcuts;
- combined prediction and Grad-CAM behind `explain_prediction()`; and
- saved explanation artifacts for the RetailMind vision module.

This closes Phase 2 with an explainable product classifier. Image segmentation and background removal are **not implemented** in this increment because they are concept-only topics for Day 8.